In [1]:
%cd /ihome/haizenstein/mez141/ondemand/SwiFUN_rsfMRI

/ihome/haizenstein/mez141/ondemand/SwiFUN_rsfMRI


/ihome/haizenstein/mez141/.conda/envs/py39/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


!ls /root/autodl-tmp/SwiFUN_clean

# Set data directory

In [ ]:
# minimal preprocessing after fMRIprep (run in the terminal)
python preprocessing_fMRIprep.py

In [14]:
# !rm -rf ./ds000030_cleaned_data

In [16]:
!ln -s /ihome/haizenstein/mez141/ondemand/DS000030_cleaned_data/* ./ds000030_cleaned_data/

In [4]:
!ls /ihome/haizenstein/mez141/ondemand/SwiFUN_clean/ds000030_cleaned_data/label/sub-10159_label.pt

/ihome/haizenstein/mez141/ondemand/SwiFUN_clean/ds000030_cleaned_data/label/sub-10159_label.pt


# mask file

In [ ]:
#run in the terminal
python generate_groupmask.py

# Run the model

In [6]:
#test
!python test/module_test_swin4d.py

img_size:  (96, 96, 96, 20)
patch_size:  (6, 6, 6, 1)
patch_dim:  (16, 16, 16, 20)
done


In [7]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA device count: 1
CUDA device name: NVIDIA A100-PCIE-40GB


In [8]:
!nvidia-smi


Thu Aug 21 07:51:52 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          On  |   00000000:C7:00.0 Off |                    0 |
| N/A   29C    P0             33W /  250W |       4MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import os
os.environ["WANDB_API_KEY"] = "bf671190f5e6fbe899fe0faf69217270eef5cd7b" 
# wandb.login(key=WANDB_API_KEY)

In [ ]:
TRAINER_ARGS="--accelerator gpu --max_epochs 20 --precision 16 --num_nodes 1" #--strategy ddp_find_unused_parameters_false" # devices should be 4 for sbatch
MAIN_ARGS='--loggername wandb --run_id base --project_name ds000030_rsfMRI_alldata_6context_6pred_13kmax --dataset_name DS000030 --image_path ./ds000030_cleaned_data --mask_input  --mask_filename group_p0.8_96.pt --save_random_split_to ./ds000030_cleaned_data/metadata/random_split42.txt'#--mask_filename harvard_oxford-wholebrain-maxprob-thr25-2mm.nii.gz' # /global/cfs/cdirs/m4244/junbeom/20249_pickled_map
DATA_ARGS='--dataset_split_num 42 --batch_size 4 --eval_batch_size 16 --num_workers 8'
DEFAULT_ARGS='--id ds000030_test_run1'
rfMRI_ARGS='--downstream_task rfMRI_next --pred_context 6 --pred_horizon 6 --k_max 13 --teacher_forcing_ratio 0'
OPTIONAL_ARGS='--cope 1 --c_multiplier 2 --clf_head_version v1 --use_scheduler --gamma 0.5 --cycle 0.7 --loss_type mse --last_layer_full_MSA True '  
RESUME_ARGS="" 

!python project/main.py $TRAINER_ARGS \
  $MAIN_ARGS \
  $DEFAULT_ARGS \
  $DATA_ARGS \
  $rfMRI_ARGS \
  $OPTIONAL_ARGS \
  $RESUME_ARGS \
  --dataset_split_num 1 \
  --seed 1 \
  --learning_rate 5e-5 \
  --model swin4d_ver7 \
  --attn_drop_rate 0.3 \
  --depth 2 2 2 \
  --embed_dim 24 \
  --sequence_length 20 \
  --swift_window_size 4 4 4 6 \
  --swift_patch_size 6 6 6 1 \
  --img_size 96 96 96 20 \
  --input_scaling_method none 

Found 135 subjects in img directory
Metadata contains 134 entries
Successfully loaded 134 valid subject-label pairs
[DEBUG in make_subject_dict] final_dict has 134 entries
[INFO] Random split saved to: ./ds000030_cleaned_data/metadata/random_split42.txt
determine_split_randomly returning: train=93, val=20, test=21

=== DS000030._set_data RESULT ===
Total samples generated: 650
Unique subjects: 93
Dataset initialized with 650 samples from 93 subjects

=== DS000030._set_data RESULT ===
Total samples generated: 140
Unique subjects: 20
Dataset initialized with 140 samples from 20 subjects

=== DS000030._set_data RESULT ===
Total samples generated: 147
Unique subjects: 21
Dataset initialized with 147 samples from 21 subjects
number of train_subj: 93
number of val_subj: 20
number of test_subj: 21
length of train_idx: 650
length of val_idx: 140
length of test_idx: 147
[rank: 0] Global seed set to 1
wandb: WARNING `wandb.require('service')` is a no-op as it is now the default behavior.
wandb: 

In [ ]:
TRAINER_ARGS="--accelerator gpu --max_epochs 20 --precision 16 --num_nodes 1" #--strategy ddp_find_unused_parameters_false" # devices should be 4 for sbatch
MAIN_ARGS='--loggername wandb --run_id base --project_name ds000030_rsfMRI_alldata_6context_6pred_13kmax --dataset_name DS000030 --image_path ./ds000030_cleaned_data --mask_input  --mask_filename group_p0.8_96.pt --save_random_split_to ./ds000030_cleaned_data/metadata/random_split42.txt'#--mask_filename harvard_oxford-wholebrain-maxprob-thr25-2mm.nii.gz' # /global/cfs/cdirs/m4244/junbeom/20249_pickled_map
DATA_ARGS='--dataset_split_num 42 --batch_size 4 --eval_batch_size 16 --num_workers 8'
DEFAULT_ARGS='--id ds000030_test_run1'
rfMRI_ARGS='--downstream_task rfMRI_next --pred_context 6 --pred_horizon 6 --k_max 13 --teacher_forcing_ratio 0.3 --tf_schedule linear'
OPTIONAL_ARGS='--cope 1 --c_multiplier 2 --clf_head_version v1 --use_scheduler --gamma 0.5 --cycle 0.7 --loss_type mse --last_layer_full_MSA True '  
RESUME_ARGS="" 

!python project/main.py $TRAINER_ARGS \
  $MAIN_ARGS \
  $DEFAULT_ARGS \
  $DATA_ARGS \
  $rfMRI_ARGS \
  $OPTIONAL_ARGS \
  $RESUME_ARGS \
  --dataset_split_num 1 \
  --seed 1 \
  --learning_rate 5e-5 \
  --model swin4d_ver7 \
  --attn_drop_rate 0.3 \
  --depth 2 2 2 \
  --embed_dim 24 \
  --sequence_length 20 \
  --swift_window_size 4 4 4 6 \
  --swift_patch_size 6 6 6 1 \
  --img_size 96 96 96 20 \
  --input_scaling_method none 